<a href="https://colab.research.google.com/github/herrrickshaw/herrrickshaw/blob/main/Stock_reporting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [24]:
symbol_arg = "" # Initialize symbol_arg
# stock_daily_report.py
# =====================
# This script generates a daily stock report for a given Indian stock symbol
# from NSE and BSE exchanges, with optional F&O data.
#
# Author: BennyThadikaran (https://github.com/BennyThadikaran?tab=repositories)
#
# Libraries:
#     pip install "nse[local]" bse
#
# Usage:
#     python stock_daily_report.py RELIANCE
#     python stock_daily_report.py TCS --output json
#     python stock_daily_report.py INFY --fno

# --- Install required libraries ---
# The 'nse[local]' library for National Stock Exchange data
# The 'bse' library for Bombay Stock Exchange data
!pip install "nse[local]" bse

# --- Standard Library Imports ---
import argparse
import json
import sys
from datetime import datetime, timedelta
from pathlib import Path

# --- Third-Party Library Imports ---
import pandas as pd # Import pandas for CSV reading and data manipulation

try:
    from nse import NSE
except ImportError:
    sys.exit("❌  Install the NSE library:  pip install 'nse[local]'")

try:
    from bse import BSE
except ImportError:
    sys.exit("❌  Install the BSE library:  pip install bse")

# --- Configuration and Constants ---
# Directory to store downloaded data and generated reports
DOWNLOAD_DIR = Path("./nse_bse_data")
DOWNLOAD_DIR.mkdir(exist_ok=True) # Create the directory if it doesn't exist

# --- Helper Functions for Formatting and Display ---
def fmt(val, prefix="₹", decimals=2):
    """Format a numeric value with optional prefix, commas, and decimal places."""
    try:
        return f"{prefix}{float(val):,.{decimals}f}"
    except (TypeError, ValueError):
        # Return original value or 'N/A' if formatting fails
        return str(val) if val else "N/A"

def pct(val):
    """Format a percentage value with an up/down arrow and color coding."""
    try:
        v = float(val)
        sign = "▲" if v >= 0 else "▼" # Unicode arrows for visual indication
        color = "\033[92m" if v >= 0 else "\033[91m" # Green for positive, Red for negative
        return f"{color}{sign} {abs(v):.2f}%\033[0m" # Reset color at the end
    except (TypeError, ValueError):
        return "N/A"

def section(title):
    """Prints a formatted section header for the report."""
    width = 60
    print(f"\n{'-' * width}")
    print(f"  {title.upper()}")
    print(f"{'-' * width}")

def row(label, value, width=28):
    """Prints a key-value pair row for display, with aligned labels."""
    print(f"  {label:<{width}} {value}")


# --- NSE Data Fetching Functions ---
def fetch_nse_quote(nse: NSE, symbol: str) -> dict:
    """Fetches a full equity quote including price band, 52-week data, etc. from NSE."""
    try:
        # 'trade_info' section often contains comprehensive real-time data
        return nse.quote(symbol, section="trade_info") or {}
    except Exception as e:
        print(f"  [NSE quote error] {e}")
        return {}

def fetch_nse_equity_quote(nse: NSE, symbol: str) -> dict:
    """Fetches OHLCV (Open, High, Low, Close, Volume) and metadata from NSE equityQuote."""
    try:
        return nse.equityQuote(symbol) or {}
    except Exception as e:
        print(f"  [NSE equityQuote error] {e}")
        return {}

def fetch_nse_meta(nse: NSE, symbol: str) -> dict:
    """Fetches company meta information (e.g., ISIN, industry) from NSE."""
    try:
        return nse.equityMetaInfo(symbol) or {}
    except Exception as e:
        print(f"  [NSE meta error] {e}")
        return {}

def fetch_nse_actions(nse: NSE, symbol: str) -> list:
    """Fetches upcoming corporate actions (dividends, splits, bonuses) for a symbol."""
    try:
        data = nse.actions(symbol=symbol) or []
        return data[:5] # Limit to top 5 actions for brevity
    except Exception as e:
        print(f"  [NSE actions error] {e}")
        return []

def fetch_nse_board_meetings(nse: NSE, symbol: str) -> list:
    """Fetches upcoming board meetings for a symbol."""
    try:
        data = nse.boardMeetings(symbol=symbol) or []
        return data[:3] # Limit to top 3 meetings
    except Exception as e:
        print(f"  [NSE board meetings error] {e}")
        return []

def fetch_nse_historical(nse: NSE, symbol: str, days: int = 30) -> list:
    """Fetches recent historical OHLCV data for a specified number of days."""
    try:
        end = datetime.today()
        start = end - timedelta(days=days)
        data = nse.fetch_equity_historical_data(
            symbol=symbol,
            from_date=start,
            to_date=end
        ) or []
        return data
    except Exception as e:
        print(f"  [NSE historical error] {e}")
        return []

def fetch_nse_option_chain(nse: NSE, symbol: str) -> dict:
    """Fetches option chain data, calculates Max Pain and Put-Call Ratio (PCR)."""
    try:
        oc = nse.optionChain(symbol)
        if oc:
            mp = nse.maxpain(oc) # Calculate Max Pain
            pcr_data = nse.compileOptionChain(oc) # Compile option chain for PCR
            return {"maxpain": mp, "pcr": pcr_data.get("pcr"), "raw": oc}
        return {}
    except Exception as e:
        print(f"  [NSE option chain error] {e}")
        return {}

def _fetch_bulk_deals_from_csv(file_path: Path) -> list:
    """Helper function to read bulk deals from a local CSV file (fallback)."""
    try:
        df = pd.read_csv(file_path)
        # Clean column names for easier access
        df.columns = [col.lower().strip() for col in df.columns]

        # Define a mapping from common CSV column names to internal keys
        column_mapping = {
            'client name': 'clientname',
            'quantity traded': 'quantity',
            'trade price / wght. avg. price': 'price',
            'buy / sell': 'buysell'
        }
        df = df.rename(columns=column_mapping)

        # Verify that all required columns are present after renaming
        required_cols = {'symbol', 'clientname', 'quantity', 'price', 'buysell'}
        if not required_cols.issubset(df.columns):
            print(f"  [Bulk deals CSV error] Missing required columns: {required_cols - set(df.columns)}")
            return []

        # Convert DataFrame rows to a list of dictionaries
        return df.to_dict(orient='records')
    except FileNotFoundError:
        print(f"  [Bulk deals CSV error] Fallback file not found at {file_path}.")
        return []
    except Exception as e:
        print(f"  [Bulk deals CSV error] Could not read CSV fallback: {e}")
        return []

def fetch_nse_bulk_deals(nse: NSE) -> list:
    """Fetches today's bulk deals from NSE, with a CSV fallback mechanism."""
    try:
        today = datetime.today()
        # The `bulkdeals` method typically requires date and option type.
        # Assuming 'ALL' for option_type to get all deals for the day.
        return nse.bulkdeals(option_type='ALL', fromdate=today, todate=today) or []
    except Exception as e:
        print(f"  [NSE bulk deals error] {e}")
        # --- Fallback Mechanism ---
        print("  Attempting to fetch bulk deals from fallback source...")
        try:
            fallback_file = DOWNLOAD_DIR / "bulk_deals.csv" # Expected fallback CSV file name
            fallback_deals = _fetch_bulk_deals_from_csv(fallback_file)

            if fallback_deals:
                print("  Fallback for bulk deals successful.")
                return fallback_deals
            else:
                print("  Fallback for bulk deals returned no data or is not implemented.")
                return []
        except Exception as fallback_e:
            print(f"  [NSE bulk deals fallback error] {fallback_e}")
            return []


# --- BSE Data Fetching Functions ---
def fetch_bse_data(bse: BSE, ticker: str) -> dict:
    """Fetches BSE scrip code, OHLC quote, and corporate actions for a given ticker."""
    result = {}
    try:
        scrip_code = bse.getScripCode(ticker)
        if not scrip_code:
            print(f"  [BSE error] Scrip code not found for {ticker}")
            return result
        result["scrip_code"] = scrip_code

        quote = bse.quote(scrip_code) or {}
        result["quote"] = quote

        actions = bse.actions(scripcode=scrip_code) or []
        result["actions"] = actions[:5] # Limit to top 5 actions
    except Exception as e:
        print(f"  [BSE error] {e}")
    return result


# --- Display Functions for Report Output ---
def display_nse_price(eq_quote: dict, trade_info: dict):
    """Displays live price and trade information from NSE."""
    section("NSE — Live Price & Trade Info")

    # Extract relevant data from nested dictionaries
    pd_info = eq_quote.get("priceInfo", {})
    md_info = eq_quote.get("metadata", {})
    sd_info = eq_quote.get("securityInfo", {})
    ti_info = trade_info.get("trade_info", {})

    # Price-related data
    ltp = pd_info.get("lastPrice") or pd_info.get("ltp")
    prev_close = pd_info.get("previousClose") or pd_info.get("prevClose")
    change = pd_info.get("change")
    pct_change = pd_info.get("pChange")
    open_ = pd_info.get("open")
    high = pd_info.get("intraDayHighLow", {}).get("max") or pd_info.get("dayHigh")
    low = pd_info.get("intraDayHighLow", {}).get("min") or pd_info.get("dayLow")
    volume = ti_info.get("totalTradedVolume") or pd_info.get("totalTradedVolume")
    value = ti_info.get("totalTradedValue") or pd_info.get("totalTradedValue")
    vwap = pd_info.get("vwap")
    w52h = pd_info.get("weekHighLow", {}).get("max")
    w52l = pd_info.get("weekHighLow", {}).get("min")
    price_band = pd_info.get("priceBand", {})
    upper_band = price_band.get("upper") if price_band else ti_info.get("priceBandHigh")
    lower_band = price_band.get("lower") if price_band else ti_info.get("priceBandLow")

    # Display company and identifier info
    row("Company",       md_info.get("companyName") or md_info.get("symbol", "—"))
    row("ISIN",          md_info.get("isin", "—"))
    row("Series",        md_info.get("series", "—"))
    row("Industry",      md_info.get("industry", "—"))
    row("Face Value",    fmt(sd_info.get("faceValue"), prefix="₹"))
    print()

    # Display current price and daily movement
    row("Last Traded Price", fmt(ltp))
    row("Prev Close",        fmt(prev_close))
    row("Change",            f"{fmt(change, prefix='₹')}  {pct(pct_change)}")
    row("Open",              fmt(open_))
    row("Day High",          fmt(high))
    row("Day Low",           fmt(low))
    row("VWAP",              fmt(vwap))
    print()

    # Display 52-week and volume data
    row("52‑Week High",  fmt(w52h))
    row("52‑Week Low",   fmt(w52l))
    row("Volume",        f"{int(volume):,}" if volume else "N/A")
    # Convert traded value to Crores (Cr) for large values
    traded_value_cr = value / 10_000_000 if value else None # 1 Crore = 10 million
    row("Traded Value",  fmt(traded_value_cr, prefix="₹", decimals=2) + " Cr" if traded_value_cr else "N/A")
    print()

    # Display price band info
    row("Price Band Upper", fmt(upper_band))
    row("Price Band Lower", fmt(lower_band))


def display_bse_price(bse_data: dict):
    """Displays live quote (OHLC) information from BSE."""
    section("BSE — Live Quote (OHLC)")
    bse_quote = bse_data.get("quote", {})
    if not bse_quote:
        print("  No BSE data available.")
        return

    # Extract and display BSE specific data
    row("BSE Scrip Code", str(bse_data.get("scrip_code", "—")))
    row("LTP",            fmt(bse_quote.get("LastRate") or bse_quote.get("ltp")))
    row("Open",           fmt(bse_quote.get("OpenRate") or bse_quote.get("open")))
    row("High",           fmt(bse_quote.get("High") or bse_quote.get("dayHigh")))
    row("Low",            fmt(bse_quote.get("Low") or bse_quote.get("dayLow")))
    row("Prev Close",     fmt(bse_quote.get("PrevRate") or bse_quote.get("prevClose")))
    row("Volume",         f"{int(bse_quote.get('Volume', 0) or 0):,}")
    # Convert market cap to Crores (Cr) for large values
    market_cap_cr = bse_quote.get("Mktcap") / 10_000_000 if bse_quote.get("Mktcap") else None
    row("Market Cap",     fmt(market_cap_cr, prefix="₹", decimals=2) + " Cr" if market_cap_cr else "N/A")


def display_corporate_actions(nse_actions: list, bse_actions: list):
    """Displays upcoming corporate actions from both NSE and BSE."""
    section("Upcoming Corporate Actions")

    print("  NSE:")
    if nse_actions:
        for action in nse_actions:
            date = action.get("exDate") or action.get("recordDate") or "—"
            purpose = action.get("purpose") or action.get("subject") or "—"
            print(f"    • {date:<14} {purpose}")
    else:
        print("    No upcoming actions found.")

    print("\n  BSE:")
    if bse_actions:
        for action in bse_actions:
            date = action.get("exDate") or action.get("Ex_Date") or "—"
            purpose = action.get("purpose") or action.get("Purpose") or "—"
            print(f"    • {date:<14} {purpose}")
    else:
        print("    No upcoming actions found.")


def display_board_meetings(meetings: list):
    """Displays upcoming board meetings."""
    section("Board Meetings")
    if not meetings:
        print("  No upcoming board meetings.")
        return
    for meeting in meetings:
        date = meeting.get("meetingDate") or meeting.get("bm_date") or "—"
        purpose = meeting.get("purpose") or meeting.get("bm_purpose") or "—"
        print(f"  • {date:<14} {purpose}")


def display_option_chain(oc_data: dict):
    """Displays a summary of option chain data including Max Pain and PCR."""
    section("F&O — Option Chain Summary")
    if not oc_data:
        print("  Not an F&O stock or data unavailable.")
        return

    max_pain = oc_data.get("maxpain")
    pcr = oc_data.get("pcr")

    row("Max Pain Strike", fmt(max_pain) if max_pain else "N/A")
    row("Put-Call Ratio (PCR)", f"{pcr:.2f}" if pcr is not None else "N/A")

    # Determine market sentiment based on PCR
    if pcr is not None:
        sentiment = "Bullish (PCR > 1)" if pcr > 1 else "Bearish (PCR ≤ 1)"
    else:
        sentiment = "N/A"
    row("Market Sentiment", sentiment)


def display_historical_summary(history: list):
    """Displays a summary of recent historical price data."""
    section("Recent Historical Data (Last 30 Days)")
    if not history:
        print("  No historical data available.")
        return

    closing_prices = []
    for record in history:
        try:
            # Try to get closing price from different possible keys
            close_price = float(record.get("CH_CLOSING_PRICE") or record.get("close") or 0)
            if close_price:
                closing_prices.append(close_price)
        except (TypeError, ValueError):
            pass # Ignore records with invalid closing prices

    if len(closing_prices) < 2: # Need at least two points to calculate change
        print("  Insufficient data.")
        return

    # Calculate summary statistics
    latest_close = closing_prices[-1]
    oldest_close = closing_prices[0]
    highest_close = max(closing_prices)
    lowest_close = min(closing_prices)
    average_close = sum(closing_prices) / len(closing_prices)
    period_return = ((latest_close - oldest_close) / oldest_close) * 100 if oldest_close else 0

    row("Days of data",      str(len(closing_prices)))
    row("Period High",       fmt(highest_close))
    row("Period Low",        fmt(lowest_close))
    row("Average Close",     fmt(average_close))
    row("Period Return",     pct(period_return))


def display_bulk_deals(deals: list, symbol: str):
    """Displays today's bulk deals filtered for the given symbol."""
    section("Today's Bulk Deals (NSE)")
    # Filter deals relevant to the queried symbol
    symbol_deals = [d for d in deals if str(d.get("symbol", "")).upper() == symbol.upper()]
    if not symbol_deals:
        print(f"  No bulk deals for {symbol} today.")
        return

    for deal in symbol_deals:
        client = deal.get("clientname") or "—"
        quantity = deal.get("quantity") or "—"
        price = fmt(deal.get("price"))
        buy_sell = deal.get("buysell") or "—"
        print(f"  • {buy_sell:<5} {quantity:>12} shares @ {price}  [{client}]")


# --- Main Report Generation Function ---
def run(symbol: str, show_fno: bool = False, output_format: str = "text"):
    """Generates and displays a daily stock report for the given symbol."""
    symbol = symbol.upper().strip()
    print(f"\n{'=' * 60}")
    print(f"  📊  DAILY STOCK REPORT — {symbol}")
    print(f"  ⏱  Generated: {datetime.now().strftime('%d %b %Y  %H:%M:%S')}")
    print(f"{'-' * 60}") # Use '-' for a separator below timestamp
    print(f"{'=' * 60}") # Ending with '=' for full border

    # Initialize a dictionary to store all fetched data for JSON output
    all_data = {"symbol": symbol, "timestamp": datetime.now().isoformat()}

    # Use context managers for NSE and BSE sessions to ensure proper cleanup
    with NSE(download_folder=str(DOWNLOAD_DIR), server=False) as nse, \
         BSE(download_folder=str(DOWNLOAD_DIR)) as bse:

        # --- Fetching NSE Data ---
        eq_quote   = fetch_nse_equity_quote(nse, symbol)
        trade_info = fetch_nse_quote(nse, symbol)
        nse_meta   = fetch_nse_meta(nse, symbol)
        nse_actions   = fetch_nse_actions(nse, symbol)
        board_meetings = fetch_nse_board_meetings(nse, symbol)
        history    = fetch_nse_historical(nse, symbol, days=30)
        bulk_deals = fetch_nse_bulk_deals(nse)

        # --- Fetching BSE Data ---
        bse_data   = fetch_bse_data(bse, symbol)

        # --- Fetching F&O Data (if requested) ---
        oc_data = {} # Option Chain data
        if show_fno:
            oc_data = fetch_nse_option_chain(nse, symbol)

    # --- Displaying the Report (Text Format) ---
    if output_format == "text":
        display_nse_price(eq_quote, trade_info)
        display_bse_price(bse_data)
        display_corporate_actions(nse_actions, bse_data.get("actions", []))
        display_board_meetings(board_meetings)
        if show_fno:
            display_option_chain(oc_data)
        display_historical_summary(history)
        display_bulk_deals(bulk_deals, symbol)

        print(f"\n{'=' * 60}")
        print("  Data sourced from NSE India & BSE India (unofficial APIs)")
        print(f"{'=' * 60}\n")

    # --- Generating JSON Output (if requested) ---
    elif output_format == "json":
        # Compile all fetched data into a single structure for JSON output
        all_data.update({
            "nse": {
                "equityQuote": eq_quote,
                "tradeInfo":   trade_info,
                "meta":        nse_meta,
                "actions":     nse_actions,
                "boardMeetings": board_meetings,
                "optionChain":  {k: v for k, v in oc_data.items() if k != "raw"}, # Exclude raw OC data from final JSON
                "history":     history,
                "bulkDeals":   [d for d in bulk_deals
                                if str(d.get("symbol", "")).upper() == symbol], # Filter bulk deals for the symbol
            },
            "bse": bse_data,
        })
        # Define output file path and write JSON
        out_file = DOWNLOAD_DIR / f"{symbol}_report_{datetime.today().strftime('%Y%m%d')}.json"
        out_file.write_text(json.dumps(all_data, indent=2, default=str))
        print(f"  📁  JSON saved → {out_file}")


# --- Command Line Interface (CLI) Setup ---
if __name__ == "__main__":
    # Filter out Jupyter/Colab specific arguments before parsing
    # This prevents argparse from misinterpreting internal Colab arguments
    filtered_argv = [sys.argv[0]] # Keep the script name itself

    i = 1
    while i < len(sys.argv):
        arg = sys.argv[i]
        # Common Colab/Jupyter arguments to ignore, e.g., kernel connection file
        if arg == '-f' and i + 1 < len(sys.argv):
            i += 2 # Consume both '-f' and the following file path
        elif arg.startswith('/root/.local/share/jupyter/runtime/kernel-'):
            i += 1 # Consume the kernel path argument
        else:
            # If it's not an internal argument, it's a user-provided one
            filtered_argv.append(arg)
            i += 1

    parser = argparse.ArgumentParser(
        description="Daily NSE + BSE stock report for a given ticker symbol."
    )
    parser.add_argument(
        "symbol",
        nargs='?', # Make symbol optional so script doesn't crash if not provided
        help="NSE ticker symbol (e.g. RELIANCE, TCS, INFY)",
    )
    parser.add_argument(
        "--fno",
        action="store_true", # Store True if flag is present
        default=False,
        help="Also fetch F&O option chain, PCR, and max pain (slower)",
    )
    parser.add_argument(
        "--output",
        choices=["text", "json"], # Restrict output choices
        default="text",
        help="Output format: 'text' (default) or 'json'",
    )

    # Parse arguments using the filtered list
    args = parser.parse_args(filtered_argv[1:]) # [1:] to exclude the script name itself from parsing

    # Provide a default symbol if none is specified by the user
    if not args.symbol:
        print("No stock symbol provided. Using default symbol 'RELIANCE'.")
        args.symbol = "RELIANCE"

    # Execute the main report generation function with parsed arguments
    run(args.symbol, show_fno=args.fno, output_format=args.output)


No stock symbol provided. Using default symbol 'RELIANCE'.

  📊  DAILY STOCK REPORT — RELIANCE
  ⏱  Generated: 28 Apr 2026  05:55:01
------------------------------------------------------------
  [NSE bulk deals error] Expecting value: line 2 column 1 (char 1)
  Attempting to fetch bulk deals from fallback source...
  Fallback for bulk deals successful.

------------------------------------------------------------
  NSE — LIVE PRICE & TRADE INFO
------------------------------------------------------------
  Company                      —
  ISIN                         —
  Series                       —
  Industry                     —
  Face Value                   N/A

  Last Traded Price            N/A
  Prev Close                   N/A
  Change                       N/A  N/A
  Open                         N/A
  Day High                     N/A
  Day Low                      N/A
  VWAP                         N/A

  52‑Week High                 N/A
  52‑Week Low                  N/A


In [ ]:
run('RELIANCE')


  📊  DAILY STOCK REPORT — RELIANCE
  ⏱  Generated: 28 Apr 2026  05:21:27
  [NSE bulk deals error] Expecting value: line 2 column 1 (char 1)
  Attempting to fetch bulk deals from fallback source...
  Fallback for bulk deals successful.

------------------------------------------------------------
  NSE — LIVE PRICE & TRADE INFO
------------------------------------------------------------
  Company                      —
  ISIN                         —
  Series                       —
  Industry                     —
  Face Value                   N/A

  Last Traded Price            N/A
  Prev Close                   N/A
  Change                       N/A  N/A
  Open                         N/A
  Day High                     N/A
  Day Low                      N/A
  VWAP                         N/A

  52‑Week High                 N/A
  52‑Week Low                  N/A
  Volume                       N/A
  Traded Value                 N/A

  Price Band Upper             N/A
  Price Band L

In [ ]:
run('RELIANCE')


  📊  DAILY STOCK REPORT — RELIANCE
  ⏱  Generated: 28 Apr 2026  05:24:32
  [NSE bulk deals error] Expecting value: line 2 column 1 (char 1)
  Attempting to fetch bulk deals from fallback source...
  Fallback for bulk deals successful.

------------------------------------------------------------
  NSE — LIVE PRICE & TRADE INFO
------------------------------------------------------------
  Company                      —
  ISIN                         —
  Series                       —
  Industry                     —
  Face Value                   N/A

  Last Traded Price            N/A
  Prev Close                   N/A
  Change                       N/A  N/A
  Open                         N/A
  Day High                     N/A
  Day Low                      N/A
  VWAP                         N/A

  52‑Week High                 N/A
  52‑Week Low                  N/A
  Volume                       N/A
  Traded Value                 N/A

  Price Band Upper             N/A
  Price Band L

In [ ]:
import pandas as pd

# Load the Excel file
stock_list_df = pd.read_excel('/content/Stock_List_NSE_BSE.xlsx')

# Display the first few rows to see the structure and column names
print("Stock List from Excel file:")
display(stock_list_df.head())

Stock List from Excel file:


,#,ISIN,Company Name,NSE Symbol,BSE Code,BSE Ticker
0,1,INE976A01021,NaN,WSTCSTPAPR,NaN,NaN
1,2,INE364U01010,Adani Green Energy Ltd,ADANIGREEN,541450.0,ADANIGREEN
2,3,INE742F01042,Adani Ports and Special Economic Zone Ltd,ADANIPORTS,512599.0,ADANIPORTS
3,4,INE814H01029,Adani Power Ltd,ADANIPOWER,533096.0,ADANIPOWER
4,5,INE647O01011,Aditya Birla Fashion and Retail Ltd,ABFRL,535755.0,ABFRL


In [ ]:
# Get unique NSE symbols, dropping any NaN values
nse_symbols = stock_list_df['NSE Symbol'].dropna().unique()

# Select a reasonable number of symbols to run reports for
# Let's take the first 5 for now to avoid excessively long execution
symbols_to_process = nse_symbols[:5]

print(f"Processing reports for the following NSE symbols: {', '.join(symbols_to_process)}")

# Loop through the selected symbols and run the report for each
# Set output_format='json' to save the data for dashboarding
for symbol in symbols_to_process:
    print(f"\n--- Running report for {symbol} ---")
    run(symbol, output_format='json')
    print(f"--- Finished report for {symbol} ---")

Processing reports for the following NSE symbols: WSTCSTPAPR, ADANIGREEN, ADANIPORTS, ADANIPOWER, ABFRL

--- Running report for WSTCSTPAPR ---

  📊  DAILY STOCK REPORT — WSTCSTPAPR
  ⏱  Generated: 28 Apr 2026  05:27:04
  [NSE bulk deals error] Expecting value: line 2 column 1 (char 1)
  Attempting to fetch bulk deals from fallback source...
  Fallback for bulk deals successful.

------------------------------------------------------------
  NSE — LIVE PRICE & TRADE INFO
------------------------------------------------------------
  Company                      —
  ISIN                         —
  Series                       —
  Industry                     —
  Face Value                   N/A

  Last Traded Price            N/A
  Prev Close                   N/A
  Change                       N/A  N/A
  Open                         N/A
  Day High                     N/A
  Day Low                      N/A
  VWAP                         N/A

  52‑Week High                 N/A
  52‑Week 

In [ ]:
run(args.symbol, show_fno=args.fno, output_format=args.output)


  📊  DAILY STOCK REPORT — RELIANCE
  ⏱  Generated: 28 Apr 2026  05:17:25
  [NSE bulk deals error] Expecting value: line 2 column 1 (char 1)
  Attempting to fetch bulk deals from fallback source...
  [Bulk deals CSV error] Fallback file not found at nse_bse_data/bulk_deals.csv.
  Fallback for bulk deals returned no data or is not implemented.

------------------------------------------------------------
  NSE — LIVE PRICE & TRADE INFO
------------------------------------------------------------
  Company                      —
  ISIN                         —
  Series                       —
  Industry                     —
  Face Value                   N/A

  Last Traded Price            N/A
  Prev Close                   N/A
  Change                       N/A  N/A
  Open                         N/A
  Day High                     N/A
  Day Low                      N/A
  VWAP                         N/A

  52‑Week High                 N/A
  52‑Week Low                  N/A
  Volume   

In [ ]:
!ls -l nse_bse_data

total 68
-rw-r--r-- 1 root root 66186 Apr 28 05:17 Bulk-Deals-21-04-2026-to-28-04-2026.csv


In [ ]:
!mv nse_bse_data/Bulk-Deals-21-04-2026-to-28-04-2026.csv nse_bse_data/bulk_deals.csv

In [ ]:
run(args.symbol, show_fno=args.fno, output_format=args.output)


  📊  DAILY STOCK REPORT — RELIANCE
  ⏱  Generated: 28 Apr 2026  05:18:26
  [NSE bulk deals error] Expecting value: line 2 column 1 (char 1)
  Attempting to fetch bulk deals from fallback source...
  [Bulk deals CSV error] Missing required columns: {'quantity', 'price', 'buysell', 'clientname'}
  Fallback for bulk deals returned no data or is not implemented.

------------------------------------------------------------
  NSE — LIVE PRICE & TRADE INFO
------------------------------------------------------------
  Company                      —
  ISIN                         —
  Series                       —
  Industry                     —
  Face Value                   N/A

  Last Traded Price            N/A
  Prev Close                   N/A
  Change                       N/A  N/A
  Open                         N/A
  Day High                     N/A
  Day Low                      N/A
  VWAP                         N/A

  52‑Week High                 N/A
  52‑Week Low                

In [ ]:
import pandas as pd
df = pd.read_csv('nse_bse_data/bulk_deals.csv')
print(df.head())

         Date      Symbol              Security Name   \
0  21-APR-2026    ACEINTEG  Ace Integrated Solu. Ltd.   
1  21-APR-2026    ACEINTEG  Ace Integrated Solu. Ltd.   
2  21-APR-2026    ACEINTEG  Ace Integrated Solu. Ltd.   
3  21-APR-2026  AGARWALTUF  Agarwal Tough Glass Ind L   
4  21-APR-2026  AGARWALTUF  Agarwal Tough Glass Ind L   

                          Client Name  Buy / Sell  Quantity Traded   \
0                      AMIT KUMAR JAIN        SELL         1,26,922   
1                      AMIT KUMAR JAIN         BUY         1,76,922   
2                        SUMAN CHEPURI        SELL           60,000   
3          GOODFLOW SECURITIES PVT LTD         BUY         1,46,400   
4  VPK GLOBAL VENTURES FUND - SCHEME 1        SELL         1,52,400   

  Trade Price / Wght. Avg. Price  Remarks   
0                           21.89        -  
1                           21.78        -  
2                           21.90        -  
3                          109.99        -  
4    

In [ ]:
print("\n--- First 5 rows of the Bulk Deals DataFrame ---")
display(df.head())

print("\n--- DataFrame Info (Column Types and Non-Null Counts) ---")
df.info()

print("\n--- Descriptive Statistics for Numeric Columns ---")
display(df.describe(include='all'))

print("\n--- Unique Symbols in Bulk Deals ---")
print(df['Symbol'].unique())

print("\n--- Count of Buy/Sell Transactions ---")
print(df['Buy / Sell'].value_counts())


--- First 5 rows of the Bulk Deals DataFrame ---


,Date,Symbol,Security Name,Client Name,Buy / Sell,Quantity Traded,Trade Price / Wght. Avg. Price,Remarks
0,21-APR-2026,ACEINTEG,Ace Integrated Solu. Ltd.,AMIT KUMAR JAIN,SELL,"1,26,922",21.89,-
1,21-APR-2026,ACEINTEG,Ace Integrated Solu. Ltd.,AMIT KUMAR JAIN,BUY,"1,76,922",21.78,-
2,21-APR-2026,ACEINTEG,Ace Integrated Solu. Ltd.,SUMAN CHEPURI,SELL,"60,000",21.90,-
3,21-APR-2026,AGARWALTUF,Agarwal Tough Glass Ind L,GOODFLOW SECURITIES PVT LTD,BUY,"1,46,400",109.99,-
4,21-APR-2026,AGARWALTUF,Agarwal Tough Glass Ind L,VPK GLOBAL VENTURES FUND - SCHEME 1,SELL,"1,52,400",109.00,-



--- DataFrame Info (Column Types and Non-Null Counts) ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 597 entries, 0 to 596
Data columns (total 8 columns):
 #   Column                           Non-Null Count  Dtype 
---  ------                           --------------  ----- 
 0   Date                             597 non-null    object
 1   Symbol                           597 non-null    object
 2   Security Name                    597 non-null    object
 3   Client Name                      597 non-null    object
 4   Buy / Sell                       597 non-null    object
 5   Quantity Traded                  597 non-null    object
 6   Trade Price / Wght. Avg. Price   597 non-null    object
 7   Remarks                          597 non-null    object
dtypes: object(8)
memory usage: 37.4+ KB

--- Descriptive Statistics for Numeric Columns ---


,Date,Symbol,Security Name,Client Name,Buy / Sell,Quantity Traded,Trade Price / Wght. Avg. Price,Remarks
count,597,597,597,597,597,597,597,597
unique,5,95,95,167,2,459,567,1
top,23-APR-2026,IRMENERGY,IRM Energy Limited,NK SECURITIES RESEARCH PRIVATE LIMITED,BUY,"9,00,000",286.97,-
freq,148,191,191,60,301,4,3,597



--- Unique Symbols in Bulk Deals ---


KeyError: 'Symbol'

In [26]:
print("\n--- Transaction Frequency for 'Buy / Sell' ---")

# Clean column names for easier access, consistent with _fetch_bulk_deals_from_csv
df.columns = [col.lower().strip() for col in df.columns]

# Define a mapping from common CSV column names to internal keys
column_mapping = {
    'client name': 'clientname',
    'quantity traded': 'quantity',
    'trade price / wght. avg. price': 'price',
    'buy / sell': 'buysell'
}
df = df.rename(columns=column_mapping)

# Now, access the renamed column 'buysell'
display(df['buysell'].value_counts())


--- Transaction Frequency for 'Buy / Sell' ---


,count
buysell,
BUY,301
SELL,296


In [27]:
print("\n--- Top 10 Companies by Transaction Count ---")
display(df['clientname'].value_counts().head(10))


--- Top 10 Companies by Transaction Count ---


,count
clientname,
NK SECURITIES RESEARCH PRIVATE LIMITED,60
JUNOMONETA FINSOL PRIVATE LIMITED,32
HRTI PRIVATE LIMITED,28
MICROCURVES TRADING PRIVATE LIMITED,28
QE SECURITIES LLP,24
IRAGE BROKING SERVICES LLP,16
ARIHANT CAPITAL MARKETS LIMITED,14
BLITZQUANT RESEARCH LLP,14
ELIXIR WEALTH MANAGEMENT PRIVATE LIMITED,12


First, create the directory if it doesn't exist:

```bash
!mkdir -p nse_bse_data1
```

Then, create the `nse_bse_data/bulk_deals.csv` file with the following content. You can copy and paste this into a new file in Colab or use `%%writefile` in a code cell:

```csv
symbol,clientName,quantity,price,buySell
RELIANCE,BULK INVESTORS LLC,150000,2600.00,BUY
RELIANCE,ABC TRADING CO,50000,2595.50,SELL
TCS,GLOBAL FUNDS LTD,20000,3800.75,BUY
INFY,HIGHNETWORTH IND,10000,1500.25,SELL
```

After creating the file, you can re-run the main script cell (`UIFrsDcQcsLL`) to see the bulk deals loaded from the CSV.